In [1]:
from pathlib import Path
import sqlite3
import json

import pandas as pd
import requests

## Part 1

In [2]:
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EXTERNAL_DIR = DATA_DIR / "external"

In [3]:
participants = pd.DataFrame({
    "id": ["001", "002", "003", "004", "005"],
    "age": [24, 31, 27, 42, 35],
    "group": ["A", "B", "A", "B", "A"],
    "score": [81, 94, 88, 91, 85]
})

In [4]:
display(participants)
participants.dtypes

,id,age,group,score
0,001,24,A,81
1,002,31,B,94
2,003,27,A,88
3,004,42,B,91
4,005,35,A,85


id         str
age      int64
group      str
score    int64
dtype: object

## Part 2

In [5]:
csv_path = RAW_DIR / "participants.csv"

participants.to_csv(
    csv_path,
    index=False
)

In [6]:
csv_path.exists()

True

## Part 3

In [7]:
loaded = pd.read_csv(csv_path)

In [8]:
display(loaded)
loaded.dtypes

,id,age,group,score
0,1,24,A,81
1,2,31,B,94
2,3,27,A,88
3,4,42,B,91
4,5,35,A,85


id       int64
age      int64
group      str
score    int64
dtype: object

When reading in the file, pandas defualted to reading '001, 002, 003, ...' as integers and so the leading zeros were dropped.

## Part 4

In [9]:
loaded = pd.read_csv(
    csv_path,
    dtype={
        "id": "string",
        "group": "category"
    }
)

In [10]:
print(loaded.dtypes)
loaded

id         string
age         int64
group    category
score       int64
dtype: object


,id,age,group,score
0,001,24,A,81
1,002,31,B,94
2,003,27,A,88
3,004,42,B,91
4,005,35,A,85


It is better represented as string as it is more clear that id can be any combination of 3 numbers.

## Part 5

In [11]:
messy = pd.DataFrame({
    "id": ["001", "002", "003", "004"],
    "score": ["81", "-999", "88", "NA"]
})

In [12]:
messy_path = RAW_DIR / "messy_scores.csv"

messy.to_csv(
    messy_path,
    index=False
)

In [13]:
messy_loaded = pd.read_csv(
    messy_path,
    na_values=["NA", "-999"]
)

In [14]:
messy_loaded
messy_loaded.dtypes
messy_loaded.isna().sum()

id       0
score    2
dtype: int64

### Why is documenting missing-value coding part of provenance?

It is important because it can be difficult to tell if a value indicates NA or an actual value in the data.

## Part 6

In [15]:
print(loaded.shape, '\n')
display(loaded.head())
print("Data types\n", loaded.dtypes, '\n')
print("NA values\n",loaded.isna().sum(), '\n')
print("Unique ids: ", loaded["id"].nunique(), '\n')
print("Number of groups: ", loaded["group"].value_counts(), '\n')

(5, 4) 



,id,age,group,score
0,001,24,A,81
1,002,31,B,94
2,003,27,A,88
3,004,42,B,91
4,005,35,A,85


Data types
 id         string
age         int64
group    category
score       int64
dtype: object 

NA values
 id       0
age      0
group    0
score    0
dtype: int64 

Unique ids:  5 

Number of groups:  group
A    3
B    2
Name: count, dtype: int64 



In [16]:
assert len(loaded) == 5
assert loaded["id"].is_unique

In [17]:
assert set(
    loaded["group"].astype(str)
) <= {"A", "B"}

## Part 7

In [18]:
db_path = RAW_DIR / "study.db"

connection = sqlite3.connect(db_path)

In [19]:
participants.to_sql(
    "participants",
    connection,
    if_exists="replace",
    index=False
)

5

## Part 8

In [20]:
sql_df = pd.read_sql(
    "SELECT * FROM participants",
    connection
)

In [21]:
sql_df

,id,age,group,score
0,001,24,A,81
1,002,31,B,94
2,003,27,A,88
3,004,42,B,91
4,005,35,A,85


In [22]:
query = """
SELECT
    id,
    age,
    score
FROM participants
"""

selected = pd.read_sql(
    query,
    connection
)

In [23]:
participants[
    ["id", "age", "score"]
]

,id,age,score
0,001,24,81
1,002,31,94
2,003,27,88
3,004,42,91
4,005,35,85


The query select only the specified columns from participants, while * selects all.

## Part 10

In [24]:
query = """
SELECT *
FROM participants
WHERE age >= 30
"""

older = pd.read_sql(
    query,
    connection
)

In [25]:
participants.loc[
    participants["age"] >= 30
]

,id,age,group,score
1,002,31,B,94
3,004,42,B,91
4,005,35,A,85


The new query only selects participants that are older than 30.

## Part 11

In [26]:
query = """
SELECT
    "group",
    COUNT(*) AS n,
    AVG(score) AS mean_score
FROM participants
GROUP BY "group"
"""

summary = pd.read_sql(
    query,
    connection
)

In [27]:
summary

,group,n,mean_score
0,A,3,84.666667
1,B,2,92.500000


In [28]:
query = """
SELECT
    "group",
    MIN(score) AS min_score,
    MAX(score) AS max_score
FROM participants
GROUP BY "group" 
"""
summary = pd.read_sql(query, connection)

summary

,group,min_score,max_score
0,A,81,88
1,B,91,94


## Part 12

In [29]:
query = """
SELECT *
FROM participants
WHERE "group" = ?
"""

group_a = pd.read_sql(
    query,
    connection,
    params=("A",)
)

group_a

,id,age,group,score
0,001,24,A,81
1,003,27,A,88
2,005,35,A,85


### Why might parameters be preferable to inserting values directly into a query string?

It might be preferable because it allows for more flexibility. Instead of needing to make a new query you can reuse the same one and just specify a new parameter.

## Part 13

In [30]:
outcomes = pd.DataFrame({
    "id": ["001", "002", "003", "005"],
    "followup_score": [85, 97, 91, 89]
})

In [31]:
outcomes.to_sql(
    "outcomes",
    connection,
    if_exists="replace",
    index=False
)

4

## Part 14

In [32]:
query = """
SELECT
    p.id,
    p.age,
    p."group",
    p.score,
    o.followup_score
FROM participants AS p
LEFT JOIN outcomes AS o
    ON p.id = o.id
"""

joined = pd.read_sql(
    query,
    connection
)

In [33]:
display(joined)
print(joined.isna().sum())

,id,age,group,score,followup_score
0,001,24,A,81,85.0
1,002,31,B,94,97.0
2,003,27,A,88,91.0
3,004,42,B,91,NaN
4,005,35,A,85,89.0


id                0
age               0
group             0
score             0
followup_score    1
dtype: int64


This is equivalent to a left join, with participants on the left. This is why we are left with an NA for 004

## Part 15

In [34]:
connection.close()

### Why close resources when they are no longer needed?

We close connect for safety. We do not want to accidently write to this database or let anyone else gain access.

## Part 16

In [35]:
url = "https://jsonplaceholder.typicode.com/users"

response = requests.get(
    url,
    timeout=10
)

In [36]:
response.status_code

200

In [37]:
response.raise_for_status()

To ensure proper connection.

## Part 17

In [38]:
response.text[:500]

'[\n  {\n    "id": 1,\n    "name": "Leanne Graham",\n    "username": "Bret",\n    "email": "Sincere@april.biz",\n    "address": {\n      "street": "Kulas Light",\n      "suite": "Apt. 556",\n      "city": "Gwenborough",\n      "zipcode": "92998-3874",\n      "geo": {\n        "lat": "-37.3159",\n        "lng": "81.1496"\n      }\n    },\n    "phone": "1-770-736-8031 x56442",\n    "website": "hildegard.org",\n    "company": {\n      "name": "Romaguera-Crona",\n      "catchPhrase": "Multi-layered client-server neural-'

In [39]:
data = response.json()

In [40]:
print(type(data))
print()
print(len(data))
print()
print(type(data[0]))
print()
print(data[0])

<class 'list'>

10

<class 'dict'>

{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address': {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}, 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}}


It becomes a list of nested dictionaries.

## Part 18

In [41]:
users = pd.DataFrame(data)

In [42]:
display(users.head())
print(users.columns)

,id,name,username,email,address,phone,website,company
0,1,Leanne Graham,Bret,Sincere@april.biz,"{'street': 'Kulas Light', 'suite': 'Apt. 556',...",1-770-736-8031 x56442,hildegard.org,"{'name': 'Romaguera-Crona', 'catchPhrase': 'Mu..."
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,"{'street': 'Victor Plains', 'suite': 'Suite 87...",010-692-6593 x09125,anastasia.net,"{'name': 'Deckow-Crist', 'catchPhrase': 'Proac..."
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,"{'street': 'Douglas Extension', 'suite': 'Suit...",1-463-123-4447,ramiro.info,"{'name': 'Romaguera-Jacobson', 'catchPhrase': ..."
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,"{'street': 'Hoeger Mall', 'suite': 'Apt. 692',...",493-170-9623 x156,kale.biz,"{'name': 'Robel-Corkery', 'catchPhrase': 'Mult..."
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,"{'street': 'Skiles Walks', 'suite': 'Suite 351...",(254)954-1289,demarco.info,"{'name': 'Keebler LLC', 'catchPhrase': 'User-c..."


Index(['id', 'name', 'username', 'email', 'address', 'phone', 'website',
       'company'],
      dtype='str')


## Part 23

In [45]:
url = (
    "https://www.basketball-reference.com/leagues/NBA_2023_per_game.html"
)

tables = pd.read_html(url)

In [47]:
tables

[        Rk                   Player   Age Team  Pos     G    GS    MP    FG  \
 0      1.0              Joel Embiid  28.0  PHI    C  66.0  66.0  34.6  11.0   
 1      2.0              Luka Dončić  23.0  DAL   PG  66.0  66.0  36.2  10.9   
 2      3.0           Damian Lillard  32.0  POR   PG  58.0  58.0  36.3   9.6   
 3      4.0  Shai Gilgeous-Alexander  24.0  OKC   PG  68.0  68.0  35.5  10.4   
 4      5.0    Giannis Antetokounmpo  28.0  MIL   PF  63.0  63.0  32.1  11.2   
 ..     ...                      ...   ...  ...  ...   ...   ...   ...   ...   
 675  536.0            Deonte Burton  29.0  SAC   PF   2.0   0.0   3.0   0.0   
 676  537.0       Michael Foster Jr.  20.0  PHI   PF   1.0   0.0   1.0   0.0   
 677  538.0            Frank Jackson  24.0  UTA   PG   1.0   0.0   5.0   0.0   
 678  539.0         Alondes Williams  23.0  BRK   SG   1.0   0.0   5.0   0.0   
 679    NaN           League Average   NaN  NaN  NaN   NaN   NaN   NaN   NaN   
 
       FGA  ...  ORB  DRB   TRB  AST  

In [46]:
print(len(tables))
tables[0].head()

2


,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards
0,1.0,Joel Embiid,28.0,PHI,C,66.0,66.0,34.6,11.0,20.1,...,1.7,8.4,10.2,4.2,1.0,1.7,3.4,3.1,33.1,"MVP-1,DPOY-9,CPOY-5,AS,NBA1"
1,2.0,Luka Dončić,23.0,DAL,PG,66.0,66.0,36.2,10.9,22.0,...,0.8,7.8,8.6,8.0,1.4,0.5,3.6,2.5,32.4,"MVP-8,CPOY-8,AS,NBA1"
2,3.0,Damian Lillard,32.0,POR,PG,58.0,58.0,36.3,9.6,20.7,...,0.8,4.0,4.8,7.3,0.9,0.3,3.3,1.9,32.2,"CPOY-10,AS,NBA3"
3,4.0,Shai Gilgeous-Alexander,24.0,OKC,PG,68.0,68.0,35.5,10.4,20.3,...,0.9,4.0,4.8,5.5,1.6,1.0,2.8,2.8,31.4,"MVP-5,CPOY-7,AS,NBA1"
4,5.0,Giannis Antetokounmpo,28.0,MIL,PF,63.0,63.0,32.1,11.2,20.3,...,2.2,9.6,11.8,5.7,0.8,0.8,3.9,3.1,31.1,"MVP-3,DPOY-6,AS,NBA1"


## Part 24

In [48]:
for i, table in enumerate(tables[:5]):
    print(
        i,
        table.shape,
        list(table.columns)[:4]
    )

0 (680, 31) ['Rk', 'Player', 'Age', 'Team']
1 (218, 31) ['Rk', 'Player', 'Age', 'Team']


In [51]:
tables[0].info()

<class 'pandas.DataFrame'>
RangeIndex: 680 entries, 0 to 679
Data columns (total 31 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Rk      679 non-null    float64
 1   Player  680 non-null    str    
 2   Age     679 non-null    float64
 3   Team    679 non-null    str    
 4   Pos     679 non-null    str    
 5   G       679 non-null    float64
 6   GS      679 non-null    float64
 7   MP      679 non-null    float64
 8   FG      679 non-null    float64
 9   FGA     679 non-null    float64
 10  FG%     677 non-null    float64
 11  3P      679 non-null    float64
 12  3PA     679 non-null    float64
 13  3P%     656 non-null    float64
 14  2P      679 non-null    float64
 15  2PA     679 non-null    float64
 16  2P%     673 non-null    float64
 17  eFG%    677 non-null    float64
 18  FT      679 non-null    float64
 19  FTA     679 non-null    float64
 20  FT%     643 non-null    float64
 21  ORB     679 non-null    float64
 22  DRB     679 n